## Classification de sentiemens dans le  texte 
Naif : "A et B sont independants "
P(A|B) = P(A) B n'as aucun influence sur A
- 1er step : 
P(Y) {P(Y=0),P(Y=1)}
- 2eme step:
Etablir vocabulaire 
["mot",count]
- 3eme step : 
P("texte1"/Y)
- 4eme step: 
P(Y/"texte") = P(Y)*P(T/Y)

In [1]:
# ── Imports ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
 
from collections import Counter
 
from sklearn.naive_bayes        import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline           import Pipeline
from sklearn.metrics            import (accuracy_score, classification_report,
                                        confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection    import cross_val_score
 
import warnings
warnings.filterwarnings("ignore")




In [2]:
def Read_txt(data):
    df = pd.read_csv(data, sep='\t', header=None, names=['sentence', 'label'],encoding='utf-8')
    df.dropna(inplace=True)
    df["label"] = df['label'].astype(int)
    return df

print("chargement des données...")
# divisioin des dataset 
test_df = Read_txt("./data/sentiment+labelled+sentences/sentiment labelled sentences/imdb_labelled.txt")
train_df = Read_txt("./data/sentiment+labelled+sentences/sentiment labelled sentences/amazon_cells_labelled.txt")

print(test_df.head(5).to_string(index=False))
print(train_df.head(5).to_string(index=False))

chargement des données...
                                                                                                                                                                                    sentence  label
                                                                                                     A very, very, very slow-moving, aimless movie about a distressed, drifting young man.        0
                                                                                         Not sure who was more lost - the flat characters or the audience, nearly half of whom walked out.        0
Attempting artiness with black & white and clever camera angles, the movie disappointed - became even more ridiculous - as the acting was poor and the plot and lines almost non-existent.        0
                                                                                                                                                Very little music or anything to speak of.    

In [3]:
## Calcul de probabilité de P(y)
count = train_df["label"].value_counts().sort_index()
total = len(train_df)
prob_y = count / total
print("\nProbabilité de P(y):")
print(prob_y)

##repartition des target dans le train 
summary = pd.DataFrame({
    "classe ": ["Y : 0 (negative)", "Y : 1 (positive)" ],
    "Nb_exemple": [count[0], count[1]],
    "Pourcentage": [prob_y[0] * 100, prob_y[1] * 100]
})
print("\nRépartition des classes dans le train:")
print(summary.to_string(index=False))


Probabilité de P(y):
label
0    0.5
1    0.5
Name: count, dtype: float64

Répartition des classes dans le train:
         classe   Nb_exemple  Pourcentage
Y : 0 (negative)         500         50.0
Y : 1 (positive)         500         50.0


## les classe sont bien reparties 


In [4]:
## Frequence des mots 
print("extraire les token")
cv_full = CountVectorizer(
    token_pattern=r"[a-z]+",
    lowercase=True,
    stop_words=None #On affiche exactement les mot comme dans l'exemple
)
# neg_senteces =  train_df[train_df["label"] == 0]["sentence"]
# pos_senteces =  train_df[train_df["label"] == 1]["sentence"]
# neg_senteces.to_list()
# print(type(neg_senteces))
# Séparer les titres par classe
pos_senteces = " ".join(train_df[train_df["label"] == 1]["sentence"])
neg_senteces = " ".join(train_df[train_df["label"] == 0]["sentence"])

# Comptage des mots
pos_senteces_words = Counter(pos_senteces.lower().split()).most_common(50)
neg_senteces_words = Counter(neg_senteces.lower().split()).most_common(50)


positive_words = Counter(pos_senteces.lower().split()).most_common(50)
negative_words = Counter(neg_senteces.lower().split()).most_common(50)
print("les mot positive",positive_words)
print("les mots negative",negative_words)


extraire les token
les mot positive [('the', 237), ('and', 188), ('i', 151), ('is', 139), ('it', 108), ('a', 104), ('this', 100), ('to', 85), ('my', 72), ('very', 69), ('for', 65), ('great', 62), ('with', 62), ('phone', 57), ('good', 53), ('of', 49), ('works', 43), ('on', 43), ('have', 37), ('was', 36), ('in', 34), ('that', 30), ('so', 26), ('has', 24), ('headset', 24), ('sound', 22), ('one', 22), ('are', 21), ('quality', 21), ('excellent', 20), ('you', 20), ('battery', 20), ('phone.', 20), ('as', 20), ('love', 20), ('but', 20), ("i've", 19), ('best', 19), ('nice', 19), ('than', 19), ('recommend', 18), ('like', 18), ('well', 17), ('all', 17), ('had', 17), ('from', 16), ("it's", 16), ('would', 16), ('great.', 15), ('any', 15)]
les mots negative [('the', 276), ('i', 162), ('it', 129), ('and', 122), ('a', 113), ('to', 110), ('not', 101), ('is', 99), ('this', 97), ('my', 71), ('of', 70), ('phone', 61), ('for', 54), ('was', 54), ('in', 53), ('that', 45), ('with', 45), ('on', 44), ('you', 42

In [5]:
##calculer la probabilité des P(Y/texte)

pipeline = Pipeline([
    ("vectoriser",CountVectorizer(
        token_pattern=r"[a-z]+",
        lowercase=True,
        stop_words=None
    )),
    ("nb",MultinomialNB(alpha=1))
])

X_train = train_df["sentence"]
y_train = train_df["label"]
X_test = test_df["sentence"]
y_test = test_df["label"]

pipeline.fit(X_train, y_train)
vectorizer= pipeline.named_steps["vectoriser"]
nb = pipeline.named_steps["nb"]
feature_names = vectorizer.get_feature_names_out()
vocab_size = len(feature_names)

print(f"taille du vocabulaire {vocab_size}")
print(f"taille du vocab {len(train_df)}")
print(f"taille {len(test_df)}")

exemple_de_mot= ["good", "bad", "great", "terrible", "love", "hate",
                "worst", "best", "amazing", "awful"]
indice_de_mot = {m: i for i, m in enumerate(feature_names)}
ligne = []
for m in exemple_de_mot:
    if m in indice_de_mot:
        idx = indice_de_mot[m]
        ligne.append({
            "Mot": m,
            "P(w|Y=0)"  : round(nb.feature_log_prob_[0, idx], 4),
            "P(w|Y=1)"  : round(nb.feature_log_prob_[1, idx], 4),

        })
        
        
df_proba = pd.DataFrame(ligne)
print(df_proba.to_string(index=False))


taille du vocabulaire 1812
taille du vocab 1000
taille 748
     Mot  P(w|Y=0)  P(w|Y=1)
    good   -6.2410   -4.6387
     bad   -6.1720   -8.8283
   great   -7.0883   -4.2745
terrible   -6.4821   -8.8283
    love   -8.8800   -5.7838
    hate   -7.2706   -8.8283
   worst   -6.1720   -8.8283
    best   -7.7814   -5.7373
 amazing   -8.8800   -8.1352
   awful   -7.4937   -8.1352


In [6]:
print("\n" + "=" * 60)
print("STEP 4 — Prédiction & Évaluation sur le jeu de test")
print("=" * 60)
 
y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)
 
acc = accuracy_score(y_test, y_pred)
 
print(f"\n  Accuracy sur le test : {acc:.4f}  ({acc*100:.2f}%)")
print(f"\n  Rapport de classification :\n")
print(classification_report(y_test, y_pred,
                             target_names=["Négatif (0)", "Positif (1)"]))
 
# ── Cross-validation   ──────────────────────────────────
# all_data = pd.concat([train_df, test_df], ignore_index=True)
# cv_scores = cross_val_score(pipeline,
#                              all_data["sentence"],
#                              all_data["label"],
#                              cv=5, scoring="accuracy")
# print(f"  Cross-validation 5-fold (toutes données) :")
# print(f"    Scores  : {np.round(cv_scores, 4)}")
# print(f"    Moyenne : {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}")
 
# cm  = confusion_matrix(y_test, y_pred)
# fig, ax = plt.subplots(figsize=(5, 4))
# disp = ConfusionMatrixDisplay(confusion_matrix=cm,
#                                display_labels=["Négatif (0)", "Positif (1)"])
# disp.plot(ax=ax, colorbar=False,
#           cmap="RdYlGn")
# ax.set_title("Step 4 — Matrice de confusion (test)", fontsize=12, fontweight="bold")
# plt.tight_layout()
# plt.savefig("step4_confusion_matrix.png", dpi=150)
# plt.show()



STEP 4 — Prédiction & Évaluation sur le jeu de test

  Accuracy sur le test : 0.6912  (69.12%)

  Rapport de classification :

              precision    recall  f1-score   support

 Négatif (0)       0.65      0.77      0.71       362
 Positif (1)       0.74      0.62      0.67       386

    accuracy                           0.69       748
   macro avg       0.70      0.69      0.69       748
weighted avg       0.70      0.69      0.69       748



In [9]:
new_sentences = [
    "This product is absolutely amazing and I love it",
    "Terrible quality, I hate this product",
    "It was okay, nothing special",
    "Best purchase I have ever made",
    "Complete waste of money, very disappointed",
    "Highly recommend, works perfectly",
    "Broke after one day, do not buy",
    "I hate this items",
    "I agree with that product",
    "wasted two hours",
    "you deserve love",
    "you deserve hate",
    "Wow... Loved this place."
]
 
preds  = pipeline.predict(new_sentences)
probas = pipeline.predict_proba(new_sentences)
 
demo_df = pd.DataFrame({
    "Phrase"      : new_sentences,
    "P(Négatif)"  : probas[:, 0].round(4),
    "P(Positif)"  : probas[:, 1].round(4),
    "Prédiction"  : ["POSITIF" if p == 1 else "NÉGATIF" for p in preds]
})
pd.set_option("display.max_colwidth", 50)
print(f"\n{demo_df.to_string(index=False)}")
 



                                          Phrase  P(Négatif)  P(Positif) Prédiction
This product is absolutely amazing and I love it      0.0028      0.9972    POSITIF
           Terrible quality, I hate this product      0.9428      0.0572    NÉGATIF
                    It was okay, nothing special      0.7845      0.2155    NÉGATIF
                  Best purchase I have ever made      0.1038      0.8962    POSITIF
      Complete waste of money, very disappointed      0.9989      0.0011    NÉGATIF
               Highly recommend, works perfectly      0.0004      0.9996    POSITIF
                 Broke after one day, do not buy      0.9975      0.0025    NÉGATIF
                               I hate this items      0.8922      0.1078    NÉGATIF
                       I agree with that product      0.3789      0.6211    POSITIF
                                wasted two hours      0.8920      0.1080    NÉGATIF
                                you deserve love      0.0762      0.9238   